# Recipe Information Retrieval System
**Author:** Ian Auger  
**Course:** INFO 624 — Information Retrieval  

---

## Abstract

Online recipe platforms suffer from a compounding information problem: a corpus of hundreds of thousands of recipes indexed against millions of uniformly positive five-star reviews. This optimism bias buries meaningful qualitative signals from Users beneath a statistically indistinguishable mass of praise.

This project constructs a multi-stage recipe search engine that addresses this problem by integrating three layers of evidence: classical lexical retrieval via Elasticsearch BM25, structured rule-based query alignment derived from parsed user intent, and semantic similarity computed against a 128-dimensional recipe embedding space learned by a residual neural network trained on review-derived quality signals.

The system is evaluated across five distinct search modes — Lexical, Semantic, Quality, Ablation (no semantic), and Hybrid, using NDCG@5 and Precision@1 over a stratified query set spanning high, medium, and low intent tiers. This evaluation framework directly isolates the marginal contribution of each architectural component.

In [ ]:
import sys
from pathlib import Path
p = Path.cwd().resolve()
while p != p.parent and not (p / "src").exists():
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))

from src.config import load_settings
import pandas as pd
import torch

s = load_settings()

from src.notebook import (
    render_mermaid,
    demo_intent_parsing,
    demo_scoring_functionality
)

df = pd.read_parquet(s.processed_recipes_path)


RecipeNet loaded from C:\Users\iauge\Documents\Drexel MSDS\INFO 624\Project\data\processed\best_model_residual_v2_all_features_mse.pth (frozen, eval mode).
RecipeNet loaded from C:\Users\iauge\Documents\Drexel MSDS\INFO 624\Project\data\processed\best_model_residual_v2_all_features_mse.pth (frozen, eval mode).


## System Context: A Three-Phase Project

This IR system represents the third and final phase of a multi-course research project. Each phase produced artifacts that feed directly into the next.

| Phase | Course | Deliverable | Role in Phase 3 |
|---|---|---|---|
| Phase 1 | DSCI 632 | Gold-labeled review dataset (65K zero-shot reviews, ~975K labeled reviews, 17-tag culinary taxonomy) | Source of `pred_*` and `intensity_*` features indexed per recipe |
| Phase 2 | CS 615 | RecipeNet Residual V2: 128D recipe embeddings + scalar quality predictions | Embedding bundle loaded by the reranker for semantic similarity and quality scoring |
| Phase 3 | INFO 624 | This IR system | Elasticsearch index + two-stage retrieval + Streamlit interface |

The key architectural insight is that Phase 1 and Phase 2 together transform raw recipe metadata and free-text reviews into a structured, semantically rich feature space. Phase 3 makes that feature space searchable.

## Architecture Overview

The system implements a standard two-stage retrieve-then-rerank pipeline, extended with embedding-based semantic similarity derived from the Phase 2 neural model.


**Stage 1** retrieves the top-K candidates from Elasticsearch using BM25 with structured filters and boosts derived from parsed query intent. 

**Stage 2** reranks those candidates by combining four scoring signals whose weights are dynamically selected based on the query's intent richness. This stratified weighting strategy will be discussed in depth later in this report.

In [2]:
render_mermaid()

## Data & Index

### Corpus

The search corpus is derived from the [Food.com Recipes and Reviews dataset](https://www.kaggle.com/datasets/shuyangli94/food-com-recipes-and-user-interactions), a public dataset containing approximately 230,000 recipes and 1.1 million user reviews. The raw dataset was processed across three phases before reaching this IR system.

**Processed recipe document schema:**

| Field | Type | Description |
|---|---|---|
| `name` | text (english analyzer) | Recipe title |
| `description_clean` | text (english analyzer) | Recipe description |
| `ingredients_clean` | text (standard analyzer) | Space-separated ingredient tokens |
| `steps_clean` | text (english analyzer) | Preparation steps |
| `tags_clean` | keyword | Food.com taxonomy tags (cuisine, dietary, occasion, etc.) |
| `minutes` | float | Preparation time |
| `n_steps` | integer | Number of preparation steps |
| `n_ingredients` | integer | Number of ingredients |
| `bayesian_rating` | float | Bayesian-smoothed aggregate rating (Phase 2 target variable) |
| `review_count` | integer | Number of reviews |
| `pred_{tag}` | boolean | Phase 1 binary tag prediction (17 tags) |
| `intensity_{tag}` | float | Phase 1 tag intensity — average similarity score where tag fired |
| `cat_{tag}` | boolean | Phase 2 binary tag from one-hot encoding recipe tags (top 100 most frequent (eg. `cat_italian`, `cat_vegan`, etc.)) |
| `ing_{tag}` | boolean | Phase 2 binary tag from one-hot encoding ingredients tags (top 100 most freqent (eg. `ing_chicken`, `ing_tomato`, etc.)) | 

**Indexing rationale:** The English analyzer on text fields applies stemming and stop-word removal, improving recall for lexical queries. Tags use the keyword type because they are categorical identifiers that should match exactly, not be tokenized. Numeric fields (minutes, ratings) are stored as float/integer for range filtering and function scoring.

## Query Design & Use Cases

The system is designed to serve three distinct user archetypes, each corresponding to a different level of structured query intent. These archetypes directly motivated the stratified weighting architecture.

* **Use Case 1** — The Constrained Cook (High Intent)
A home cook with specific dietary requirements, cuisine preferences, and time constraints. Their query is explicit: *"low carb italian beef skillet"* or *"vegan thanksgiving sides under 60 mins."* The system should satisfy hard constraints first (dietary filters, time limits) and reward exact structured matches (cuisine tags, protein presence) most heavily.

* **Use Case 2** — The Weeknight Planner (Medium Intent)
A user with a partial idea such as a protein, a rough time budget, or a course in mind, but no strong constraints. Query: *"quick chicken dish"* or *"italian pasta dinner."* The User might have something they need to use in the fridge, or a vague idea of the meal they want. The system balances lexical relevance with structured alignment, and the embedding layer catches semantically relevant results that rule-matching alone would miss.

* **Use Case 3** — The Exploratory Browser (Low Intent)
A user seeking inspiration rather than a specific recipe. Query: *"something cozy and warming"* or *"impressive but not too hard."* Structured intent parsing fires on nothing. The embedding similarity signal carries the load, surfacing recipes that share the query's latent semantic neighborhood even without token overlap.

Within each use case, the system adjusts the weighting schema of the inputs to the scoring calculation to adjust dynamically to the amount of discernible intent supplied within the query. 

In [3]:
# Example showing intent parsing and tier assignment for different user queries
use_cases = [
    ("High Intent",   "low carb italian beef skillet"),
    ("Medium Intent", "quick chicken dish"),
    ("Low Intent",    "something cozy and warming"),
]

demo_intent_parsing(use_cases)



USE CASE: High Intent
Query:    'low carb italian beef skillet'
Tier:     high
Weights:  lex=1.0  align=1.5  sem=0.1  quality=0.25
Parsed intent:
  proteins             ['beef']
  dietary_tags         ['low-carb']
  cuisines             ['italian']

USE CASE: Medium Intent
Query:    'quick chicken dish'
Tier:     medium
Weights:  lex=1.0  align=1.0  sem=0.5  quality=0.25
Parsed intent:
  proteins             ['chicken']

USE CASE: Low Intent
Query:    'something cozy and warming'
Tier:     low
Weights:  lex=0.75  align=0.25  sem=1.5  quality=0.5
Parsed intent:


## Ranking Architecture

### Four Scoring Signals

The reranker combines four signals whose weights are dynamically selected based on query intent richness.

| Signal | Source | Description |
|---|---|---|
| `base_score` | Elasticsearch BM25 | Lexical relevance from Stage 1 retrieval. Rewards term frequency, penalizes document length. |
| `alignment_score` | Rule-based intent matching | Rewards exact matches between parsed query intent (cuisine, protein, dietary tags, methods) and indexed recipe tags/fields. The primary signal for high-intent queries. |
| `semantic_sim` | RecipeNet cosine similarity | Cosine similarity between the query embedding (projected into the 128-D RecipeNet latent space) and the recipe's stored embedding from Phase 2. The primary signal for low-intent queries. |
| `quality_score` | RecipeNet scalar prediction | Predicted Bayesian rating from the Phase 2 Residual V2 model. Query-independent evidence of recipe quality and somewhat comparable to PageRank in web search. |

### Stratified Weight Profiles

Rather than fixed weights, the system selects a weight profile based on the count of structured intent signals extracted from the query. This allows the system to be flexible and not rely too heavily on strict rule-based scoring logic, dynamically applying semantic weighting as needed to account for any lack of structured signal. 

| Tier | Condition | w_lex | w_align | w_sem | w_quality |
|---|---|---|---|---|---|
| High | 3+ structured signals | 1.0 | 1.5 | 0.1 | 0.25 |
| Medium | 1–2 structured signals | 1.0 | 1.0 | 0.5 | 0.25 |
| Low | 0 structured signals | 0.75 | 0.25 | 1.5 | 0.5 |

### Search Modes

* **LEXICAL:** Elasticsearch BM25 only.  No reranking signal applied, raw Stage 1 ranking is returned as-is.  Serves as the baseline for all comparisons.

* **SEMANTIC:** Embedding cosine similarity + quality score only.  Alignment is zeroed out, so structured rule matching plays no role.  Isolates the pure latent-space signal from the Phase 2 RecipeNet embeddings.

* **QUALITY:** Quality score (predicted Bayesian rating) only. No lexical, alignment, or semantic signal. Surfaces what the Phase 2 model considers the best recipes regardless of query relevance.  
    
* **ABLATION_NO_SEM:** Full hybrid pipeline minus the embedding similarity term. Uses stratified weights based on query intent richness. Direct comparison with HYBRID isolates the contribution of the embedding layer.

* **HYBRID:** Full pipeline: lexical + alignment + semantic + quality, with stratified weights determined by query intent richness.

In [4]:
from src.engine import SearchMode

demo_queries = [
    "low carb italian beef skillet",
    "quick chicken dish",
    "something cozy and warming",
]

demo_scoring_functionality(demo_queries, SearchMode.HYBRID)


QUERY : low carb italian beef skillet
TIER  : high  |  lex=1.0  align=1.5  sem=0.1  quality=0.25
-----------------------------------------------------------------
  1. [27.113] Zucchini Lasagna  Lasagne    Low Carb (60.0m)  align=2.75  sim=0.956  quality=4.68
  2. [26.680] Easy Italian Beef Stew  Crock Pot (615.0m)  align=2.75  sim=0.972  quality=4.65
  3. [26.498] Italian Beef Sandwiches Ii (215.0m)  align=2.75  sim=0.957  quality=4.64
  4. [26.370] Italian Countryside Beef Soup For The Crock Pot (265.0m)  align=2.75  sim=0.912  quality=4.56
  5. [26.348] Braised Italian Beef Stew (195.0m)  align=2.75  sim=0.977  quality=4.78

QUERY : quick chicken dish
TIER  : medium  |  lex=1.0  align=1.0  sem=0.5  quality=0.25
-----------------------------------------------------------------
  1. [19.175] Quick Hot Dish (35.0m)  align=2.25  sim=0.907  quality=4.47
  2. [17.734] Chicken Dish (0.0m)  align=1.50  sim=0.737  quality=4.34
  3. [17.411] Chicken Main Dish (100.0m)  align=1.50  sim=0.936 

## Formal Evaluation

### Design

The evaluation runs all five search modes across a stratified set of 10 queries grouped into three intent tiers (4 high, 3 medium, 3 low). Every mode ranks the same Stage 1 candidate pool ensuring score differences are attributable solely to the ranking function.

**Relevance scoring** uses a rule-based proxy scorer (`score_result`) that rewards hard constraint satisfaction, structured intent matches, and lexical token overlap in recipe name and ingredients. This scorer is intentionally independent of the reranker's own alignment logic to avoid circular self-evaluation.

**Metrics:**

- **NDCG@5** (Normalized Discounted Cumulative Gain): Position-weighted ranking quality. Rewards relevant results appearing higher in the list.

- **Precision@1**: Whether the top result is relevant. Evaluates the system's ability to surface a good result immediately.

**Known limitation:** The proxy scorer shares vocabulary with the alignment signal, which means high-intent queries will show inflated scores across all modes that surface on-topic results. The *delta between modes* is more meaningful than absolute values, particularly the HYBRID vs ABLATION_NO_SEM comparison which isolates the embedding layer's contribution.

In [5]:
# Run full five-mode ablation evaluation
# Note: takes approximately 60 seconds
from src.evaluate import evaluate_engine, print_summary

summary = evaluate_engine(top_k=5)
print_summary(summary)

RecipeNet loaded from C:\Users\iauge\Documents\Drexel MSDS\INFO 624\Project\data\processed\best_model_residual_v2_all_features_mse.pth (frozen, eval mode).
Running evaluation: 10 queries x 5 modes

  Evaluating: easy chicken dinner under 45 mins
  Evaluating: vegan thanksgiving sides
  Evaluating: low carb italian beef skillet
  Evaluating: spicy pork tacos
  Evaluating: quick healthy breakfast under 15 mins
  Evaluating: quick chicken dish
  Evaluating: italian pasta dinner
  Evaluating: something cozy and warming
  Evaluating: weeknight comfort food
  Evaluating: impressive but not too hard

TABLE 1: NDCG@5 PER QUERY
Query                                  Tier             lexical        semantic         quality ablation_no_sem          hybrid
------------------------------------------------------------------------------------------------------------------------------
easy chicken dinner under 45 mins      high               0.985           0.932           0.932           1.000       

### Results Analysis

**HYBRID outperforms LEXICAL overall (+0.014 NDCG@5).** The combined reranking pipeline consistently produces better-ordered results than BM25 alone across all intent tiers, validating the two-stage architecture. The improvement is most pronounced in the medium-intent tier, where HYBRID and ABLATION_NO_SEM both achieve a perfect mean NDCG of 1.000 compared to LEXICAL's 0.969, a delta of +0.031 driven entirely by the alignment signal surfacing correctly ordered results that lexical scoring missed.

**Alignment drives the majority of the improvement.** The HYBRID vs ABLATION_NO_SEM overall delta is effecitvely zero meaning the embedding layer produces no measurable NDCG improvement over the rule-based pipeline when averaged across all queries and tiers. The root cause is the sparse query projection: `QueryFeatureProjector` activates at most a handful of binary features out of 210, producing query vectors that compress cosine similarity scores into a narrow range. This provides insufficient differentiation between candidates for the embedding to alter rankings meaningfully.

**The embedding shows query-specific value that aggregate metrics obscure.** On "low carb italian beef skillet" (high intent), SEMANTIC mode actually outperforms HYBRID — 0.995 vs 0.974 — the only query where pure embedding signal beats the full pipeline. On "something cozy and warming" (low intent), SEMANTIC achieves a perfect 1.000 while LEXICAL scores 0.907, demonstrating that the embedding correctly promotes semantically coherent results when alignment provides no signal. These individual query patterns confirm the stratification hypothesis qualitatively, even though the effect does not aggregate into tier-level NDCG gains.

**"Impressive but not too hard" is the most diagnostic query.** With zero parseable structured intent, SEMANTIC scores 0.739 and QUALITY scores 0.739 are both well below LEXICAL's 0.936 and ABLATION_NO_SEM's 0.978. The sparse projection cannot encode the semantic content of this query into a meaningful embedding position, and the quality signal is query-agnostic by design. This query exposes the ceiling of the current embedding approach: without a learned dense query encoder, low-intent exploratory queries remain primarily served by lexical retrieval.

**SEMANTIC and QUALITY consistently underperform HYBRID.** Both score +0.055 and +0.056 below HYBRID overall respectively, confirming that neither the embedding nor the quality signal alone substitutes for structured alignment on intent-bearing queries.

**Precision@1 is uniformly 1.0 across all modes and queries.** Every mode reliably surfaces a relevant result at rank 1, confirming that Stage 1 Elasticsearch retrieval is effective at populating the candidate pool. The evaluation challenge is entirely one of ordering within an already-relevant candidate set, and the next step is to provide human applied relevance judgements to query responses to achieve a more precise scoring framework.

## Index Location

The recipe corpus is indexed in a local Elasticsearch instance provisioned via Docker Desktop, running Elasticsearch 8.12.0 in single-node mode with security disabled for local development. 

The Docker configuration is defined in `docker-compose.yml` at the repository root and can be started with `docker compose up -d`. The index is populated by running `python -m src.indexer`, which creates the mappings and bulk-ingests the processed recipe documents via the Elasticsearch bulk API.

In [6]:
# Index location and summary
assert s.es_client is not None, "Elasticsearch client is not configured"
hosts = [str(node.base_url) for node in s.es_client.transport.node_pool.all()]
print(f"Elasticsearch host : {', '.join(hosts)}")
print(f"Index name         : {s.index_name}")

stats = s.es_client.indices.stats(index=s.index_name)
total = stats['indices'][s.index_name]['total']
print(f"Document count     : {total['docs']['count']:,}")
print(f"Index size         : {total['store']['size_in_bytes'] / 1e6:.1f} MB")

mapping = s.es_client.indices.get_mapping(index=s.index_name)
fields = mapping[s.index_name]['mappings']['properties']
print(f"Indexed fields     : {len(fields)}")

Elasticsearch host : http://localhost:9200
Index name         : recipes_v1
Document count     : 173,550
Index size         : 266.0 MB
Indexed fields     : 70


## Experiences & Future Work

### What Worked

The two-stage pipeline architecture proved highly effective for the recipe domain. Elasticsearch's BM25 with structured filters and soft boosts reliably surfaced on-topic candidates, and the rule-based alignment layer added meaningful precision for users with explicit constraints. The stratified weight profiles was the most conceptually satisfying design decision: it reflects a genuine property of the system (embedding similarity compresses on high-intent queries) and produces more coherent rankings across the query spectrum than a fixed-weight approach would.

The Phase 1 and Phase 2 artifacts integrated cleanly. Loading the RecipeNet embedding bundle and reconstructing the model for query encoding was straightforward once the architecture constants were verified against the saved weights. The `column_mapping.json` produced in Phase 2 served as an effective schema contract between the modeling and retrieval systems.

### What Didn't Work as Expected

The embedding-based semantic similarity did not produce measurable NDCG improvement over the rule-based pipeline in formal evaluation. The root cause is the sparse query projection: the `QueryFeatureProjector` activates at most a handful of binary features (e.g. `cat_chicken`, `cat_italian`) out of 210, producing a query vector that lands near the origin of the embedding space. This causes cosine similarity scores to compress into a narrow range, providing insufficient differentiation between candidates.

### Future Work

Four directions could meaningfully advance this system:

**1. Learned query encoder.** Replace the sparse projection with a dense query encoder trained end-to-end to map free-text queries into the RecipeNet embedding space. This is the approach taken by Dense Passage Retrieval (DPR) and would close the geometric gap between query and document representations that currently limits the embedding signal.

**2. Human relevance judgments and Learning to Rank.** Collect human-labeled relevance assessments for a representative query set and use them to train a proper LTR model (e.g. LambdaMART). This would replace the hand-tuned weight profiles with learned weights optimized directly for NDCG, and would enable independent validation of each scoring signal's contribution.

**3. Aspect-Based Sentiment Analysis.** The Word2Vec centroid classifier struggles to isolate minority instructional tags (e.g. `too_acidic`, `too_spicy`) due to conversational hubness and class imbalance. Integrating POS-tagged Noun-Adjective pairs into the tag taxonomy would improve the precision of these qualitative signals and produce cleaner phase 1 features for downstream retrieval.

**4. Revised Deep Learning Modeling.** Retrain the Residual V2 model with a loss function that penalizes errors on low-rated recipes to address the positive skew in the training distribution (mean Bayesian rating ~4.2) that currently compresses quality predictions below the 3.5 threshold. This would sharpen both the `get_quality_score` signal in the reranker and the geometric separation between high and low-quality regions of the embedding manifold.

### Personal Note

Beyond these technical improvements, my longer-term goal is to extend this framework into a personal project: a recipe tracking and recommendation app for friends and family. This would layer user preference modeling on top of the existing retrieval infrastructure, adding persistent storage, a user-facing interface, and eventually a recommender system component. The three phases of this project have built a foundation I'm genuinely excited to continue building on.

## Code & Interface

### Source Code
The complete source code for this project is organized as follows:

| Module | Description |
|---|---|
| `src/config.py` | Settings and path resolution |
| `src/indexer.py` | Elasticsearch index creation and bulk ingestion |
| `src/search.py` | Stage 1 lexical retrieval and query intent parsing |
| `src/query_encoding.py` | Structured query projection into the feature space |
| `src/reranker.py` | Stage 2 semantic reranking with stratified weighting |
| `src/engine.py` | Search mode orchestration and ablation framework |
| `src/evaluate.py` | Five-mode ablation evaluation with NDCG@5 and P@1 |
| `src/models.py` | RecipeNet architecture (Residual V2, imported from Phase 2) |
| `src/layers.py` | Neural network building blocks — FC, Residual, PLQP layers |
| `scripts/generate_umap.py` | One-time UMAP projection and reducer generation |

### CLI Entrypoint

All system components are accessible through `main.py` which acts as the CLI entrypoint:

```bash
python -m main
```

This launches an interactive menu:

```
1. Run data ingestion pipeline      # Create ES index and load recipes
2. Run search engine demo           # Query the system interactively
3. Demo intent parsing              # Inspect how a query is parsed and projected
4. Run evaluation                   # Five-mode ablation study (NDCG@5, P@1)
5. Exit
```

**Recommended first run sequence:**
```
Option 1: index the corpus (takes 1-3 minutes)
Option 2: run a search query
Option 3: inspect intent parsing
Option 4: run the full evaluation — 10 queries stratified across high, medium, and low intent tiers (~60 seconds)
```